# 请求改写

**常见用法**：注入模型不知道/不该知道的参数（API 密钥、登录态、租户 ID）、
参数标准化（别称→标准值）、结果脱敏——模型填什么都不算数，或补全它给不出的值。

**钩子内的做法**：
- `new_tc = {**tc, "args": {**tc["args"], ...}}`——外层保 name/id（tool_call_id 绑定不丢），内层保其他参数
- `request.override(tool_call=new_tc)` 生成新 request 再交给 `execute`（不可变替换，不污染原 request）
- **改写（覆盖）与校验（拒绝）语义二选一**：覆盖会让工具内的权限校验变死代码；
  身份类参数建议藏出 schema（模型无感知）或交给工具校验拒绝

In [19]:
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt.tool_node import ToolCallRequest
from rich import print

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

API_KEYS: dict[str, str] = {"WEATHER_API_KEY": "sk-123456"}  # 模拟环境变量中的密钥


@tool
def query_weather(city: str, api_key: str) -> str:
    """查询城市天气（api_key 由系统统一注入，模型无需填写）"""
    if api_key != API_KEYS.get("WEATHER_API_KEY"):
        return "认证失败"
    return f"{city}：晴，25°C"


def inject_api_key(request: ToolCallRequest, execute):
    """请求改写：所有调外部服务的工具共用——密钥只在服务端流动，模型填什么都不算数"""
    tc = request.tool_call
    new_tc = {**tc, "args": {**tc["args"], "api_key": API_KEYS.get("WEATHER_API_KEY")}}
    return execute(request.override(tool_call=new_tc))


model_with_tools = model.bind_tools([query_weather])
tool_node = ToolNode([query_weather], wrap_tool_call=inject_api_key)


class ChatState(MessagesState):
    pass


def llm_node(state: ChatState) -> dict:
    return {"messages": [model_with_tools.invoke(state["messages"])]}


def router(state: ChatState) -> Literal["tool_node", END]:
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, [END, "tool_node"])
builder.add_edge("tool_node", "llm_node")
graph = builder.compile(checkpointer=InMemorySaver())

res = graph.invoke(
    {"messages": [HumanMessage("上海天气怎么样？")]},
    config={"configurable": {"thread_id": "1"}})

print(res)

{
    'messages': [
        HumanMessage(
            content='上海天气怎么样？',
            additional_kwargs={},
            response_metadata={},
            id='f82a3056-f623-4c3b-9108-6d0f3764f77d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 38,
                    'prompt_tokens': 293,
                    'total_tokens': 331,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 165
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': 'f8761e16-82d1-40f7-bd35-98ee83b12be4',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b0ad-cdc5-76a1-b014-7bd208a199a7-0',
            tool_calls=[
                {
                    'name': 'query_weather',
                    'args': {'city': '上海'},
                    'id': 'call_00_yxfFSmiKjjBDxvMEIizW1781',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 293,
                'output_tokens': 38,
                'total_tokens': 331,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='上海：晴，25°C',
            name='query_weather',
            id='a8bc774c-96b9-495b-b8a8-a6f8ca902f55',
            tool_call_id='call_00_yxfFSmiKjjBDxvMEIizW1781'
        ),
        AIMessage(
            content='上海目前是晴天，气温 25°C，天气很不错，适合外出活动。☀️',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 21,
                    'prompt_tokens': 350,
                    'total_tokens': 371,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 222
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '53b3b4f0-9258-4382-ad06-cea4864dd1e0',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a0b0ad-d079-7121-a146-ce90c9929911-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 350,
                'output_tokens': 21,
                'total_tokens': 371,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        )
    ]
}